# 06 — Developer Experience Analysis

This notebook addresses **RQ5**: *How do the two stacks compare on developer-facing
costs — lines of code, build time, test execution time, and container size?*

We compute a composite devex score using weighted normalised metrics and produce
the radar chart and summary table used in the dissertation's recommendations chapter.

## Imports and setup

In [ ]:
import glob
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('..') / 'scripts'))
from utils import SERVICE_LABELS, SERVICE_COLORS, set_plot_style
from generate_charts import fig07_devex_radar
from generate_tables import table04_devex

set_plot_style()
%matplotlib inline

RESULTS_DIR = Path('../../benchmarks/results')

## Load devex JSON

We use the most recent `devex-*.json` produced by `benchmarks/devex/measure.sh`.
If no JSON is found (e.g. the measure script has only been run with `.txt` output),
placeholder values are substituted so the analysis can still run end-to-end.

In [ ]:
devex_files = sorted(glob.glob(str(RESULTS_DIR / 'devex-*.json')))
if devex_files:
    with open(devex_files[-1]) as fh:
        devex = json.load(fh)
    print('Loaded:', devex_files[-1])
else:
    print('No devex JSON found -- using placeholder data')
    devex = {
        'loc_src':       {'dotnet': 800,  'go': 600},
        'build_time_s':  {'dotnet': 8.0,  'go': 3.0},
        'test_time_s':   {'dotnet': 5.0,  'go': 2.0},
        'image_size_mb': {'dotnet': 120,  'go': 15},
        'test_count':    {'dotnet': 20,   'go': 10},
    }

pd.DataFrame(devex).T

## Comparison table

A side-by-side view with absolute values and the winner for each metric.
Lower is better for LOC, build/test time, and image size; higher is better
for test count.

In [ ]:
lower_better = {'loc_src', 'build_time_s', 'test_time_s', 'image_size_mb'}

rows = []
for metric, vals in devex.items():
    dn = vals.get('dotnet')
    go = vals.get('go')
    if isinstance(dn, (int, float)) and isinstance(go, (int, float)):
        if metric in lower_better:
            winner = '.NET' if dn < go else 'Go' if go < dn else 'TIE'
        else:
            winner = '.NET' if dn > go else 'Go' if go > dn else 'TIE'
        delta = f'{dn - go:+.1f}'
    else:
        winner = delta = 'n/a'
    rows.append([metric.replace('_', ' '), dn, go, delta, winner])

pd.DataFrame(rows, columns=['Metric', '.NET', 'Go', 'Delta (.NET - Go)', 'Winner'])

## Normalise to 0–1 scale

All metrics are normalised so that **lower = better** across the board
(test_count is inverted since more tests is preferable).
This shared scale allows fair radar chart comparison and composite score calculation.

In [ ]:
def normalise(dn_val, go_val, invert=False):
    lo, hi = min(dn_val, go_val), max(dn_val, go_val)
    if hi == lo:
        return 0.5, 0.5
    dn_n = (dn_val - lo) / (hi - lo)
    go_n = (go_val - lo) / (hi - lo)
    if not invert:
        return dn_n, go_n
    return 1 - dn_n, 1 - go_n

normed = {}
for metric, vals in devex.items():
    invert = metric == 'test_count'
    dn_n, go_n = normalise(vals['dotnet'], vals['go'], invert=invert)
    normed[metric] = {'dotnet': dn_n, 'go': go_n}

pd.DataFrame(normed).T.rename(columns={'dotnet': '.NET (norm)', 'go': 'Go (norm)'})

## Figure 7: developer experience radar chart

In [ ]:
fig07_devex_radar()

## Composite devex score

We compute a weighted average of normalised metrics.  The weights reflect the
relative importance of each dimension for a typical team building an EDA service:

| Metric | Weight | Rationale |
|---|---|---|
| LOC | 0.30 | Less code = lower maintenance burden and fewer defect sites |
| Build time | 0.20 | Slow builds impede inner-loop iteration |
| Test count | 0.10 | More tests improves confidence; secondary metric |
| Test time | 0.10 | Fast tests enable TDD; complements build time |
| Image size | 0.30 | Container size determines pull time, attack surface, and registry cost |

In [ ]:
WEIGHTS = {
    'loc_src':       0.30,
    'build_time_s':  0.20,
    'test_count':    0.10,
    'test_time_s':   0.10,
    'image_size_mb': 0.30,
}

score_dn = sum(normed[m]['dotnet'] * w for m, w in WEIGHTS.items() if m in normed)
score_go = sum(normed[m]['go']     * w for m, w in WEIGHTS.items() if m in normed)

print(f'.NET composite devex score: {score_dn:.4f}  (lower = better)')
print(f'Go  composite devex score: {score_go:.4f}  (lower = better)')
winner = '.NET' if score_dn < score_go else 'Go'
print(f'Winner: {winner}')

## Table 4: devex summary

In [ ]:
table04_devex()

## Interpretation

**Fill in after running with real data.**

Template:

> **Go wins on image size and build time** — a distroless binary image at ~15 MB
> vs ~120 MB for ASP.NET alpine is an 8× size reduction, directly reducing registry
> storage, deployment pull time, and attack surface.
>
> **.NET wins on test count and ecosystem richness** — MediatR's pipeline behaviour
> abstraction enables X additional integration test scenarios with less boilerplate.
>
> **Composite devex score**: Go scores A vs .NET's B — a C% lower (better) score,
> driven primarily by the image size and build time advantages.
>
> **Dissertation answer (RQ5):** Go has a lower operational devex overhead; .NET
> has richer tooling.  The recommended stack depends on team context (see Table 5).